# LeetCode 2047 — Number of Valid Words in a Sentence bằng Python

Link đề: https://leetcode.com/problems/number-of-valid-words-in-a-sentence/ · Độ khó: **Easy** · Chủ đề: String

```python
class Solution:
    def countValidWords(self, sentence: str) -> int:

```

**Mục tiêu học**

1. Đọc kỹ một đề **nhiều luật** và biến mỗi luật thành một câu `if` kiểm tra được.
2. Nắm `split()` khi có **nhiều dấu cách liên tiếp**, các hàm `isdigit`, `isalpha`/`islower`, `count`, `index`, kiểm tra ký tự bằng `in "!.,"`.
3. Viết hàm `is_valid(token)` rõ ràng (tách hàm con), trace từng luật.
4. Biết thêm 2 cách: **biểu thức chính quy** (regex) và **máy trạng thái** quét một lần.
5. Test kỹ các ca biên: `"-"`, `"."`, `"a-b."`, `"a-."`, `"-a"`, `"a!b"`.

## 1. Đề bài yêu cầu gì?

Câu chỉ gồm chữ thường `a–z`, chữ số `0–9`, gạch nối `-`, dấu câu `! . ,` và dấu cách. Câu được chia thành **token** bởi **một hoặc nhiều** dấu cách.

Token là **từ hợp lệ** nếu thỏa **cả ba** luật:

1. **Không có chữ số** (chỉ chữ thường, gạch nối, dấu câu).
2. **Tối đa một** gạch nối `-`; nếu có, hai bên nó phải là **chữ cái thường** (`"a-b"` đúng; `"-ab"`, `"ab-"` sai).
3. **Tối đa một** dấu câu; nếu có, phải ở **cuối token** (`"ab,"`, `"cd!"`, `"."` đúng; `"a!b"`, `"c.,"` sai).

Ví dụ từ hợp lệ: `"a-b."`, `"afad"`, `"ba-c"`, `"a!"`, `"!"`. Trả về **số token hợp lệ**.

| Ví dụ | sentence | Output | Ghi chú |
|---|---|---|---|
| 1 | `"cat and  dog"` | 3 | có 2 dấu cách liền nhau |
| 2 | `"!this  1-s b8d!"` | 0 | `!this`: dấu câu không ở cuối; `1-s`, `b8d!`: có chữ số |
| 3 | `"alice and  bob are playing stone-game10"` | 5 | `stone-game10` có chữ số |

Constraints: `1 <= len(sentence) <= 1000`, chỉ chứa các ký tự nêu trên, có ít nhất 1 token.

In [1]:
print("cat and  dog".split())
print("!this  1-s b8d!".split())

['cat', 'and', 'dog']
['!this', '1-s', 'b8d!']


## 2. Nền tảng Python

### 2.1. `split()` không tham số gộp nhiều dấu cách

Đây là điểm sống còn: `"cat and  dog".split(" ")` sinh token rỗng `''` — token rỗng **không phải từ**, và nếu hàm kiểm tra của ta lỡ coi `''` là hợp lệ thì đếm sai.

In [2]:
print("cat and  dog".split(" "))   # ['cat', 'and', '', 'dog']
print("cat and  dog".split())      # ['cat', 'and', 'dog']
print("   a   ".split())           # ['a']

['cat', 'and', '', 'dog']
['cat', 'and', 'dog']
['a']


### 2.2. Kiểm tra loại ký tự

| Biểu thức | Ý nghĩa | Ví dụ |
|---|---|---|
| `ch.isdigit()` | là chữ số | `'8'.isdigit()` → True |
| `ch.isalpha()` | là chữ cái | `'a'.isalpha()` → True |
| `'a' <= ch <= 'z'` | chữ thường a–z (so sánh ký tự theo mã) | `'q'` → True |
| `ch in "!.,"` | là một trong các dấu câu | `','` → True |

`in` trên chuỗi kiểm tra "ký tự/chuỗi con có nằm trong chuỗi không".

In [3]:
for ch in "a8-!.,z":
    print(repr(ch), "digit" if ch.isdigit() else "", "letter" if 'a' <= ch <= 'z' else "",
          "punct" if ch in "!.," else "", "hyphen" if ch == "-" else "")

'a'  letter  
'8' digit   
'-'    hyphen
'!'   punct 
'.'   punct 
','   punct 
'z'  letter  


### 2.3. `count`, `index`, `any`

- `tok.count("-")`: số lần `-` xuất hiện.
- `tok.index("-")`: vị trí **đầu tiên** của `-` (báo lỗi nếu không có; dùng `find` để nhận `-1` thay vì lỗi).
- `any(cond for ch in tok)`: `True` nếu **có ít nhất một** ký tự thỏa điều kiện.

In [4]:
tok = "stone-game10"
print(tok.count("-"), tok.index("-"), tok.find("?"))
print(any(ch.isdigit() for ch in tok))
print(sum(ch in "!.," for ch in "c.,"))     # đếm dấu câu: 2

1 5 -1
True
2


### 2.4. Chỉ số âm và kiểm tra hai bên

`tok[-1]` là ký tự cuối. Với vị trí gạch nối `i`, hai bên là `tok[i-1]` và `tok[i+1]`. Phải đảm bảo `0 < i < len(tok)-1` trước khi đọc, nếu không `tok[i-1]` với `i = 0` sẽ là `tok[-1]` (ký tự cuối!) — một bẫy rất khó thấy.

In [5]:
tok = "-ab"
i = tok.index("-")
print("i =", i, "| tok[i-1] =", repr(tok[i - 1]), "<- chính là ký tự CUỐI, không phải 'bên trái'!")
print("0 < i < len(tok)-1 ?", 0 < i < len(tok) - 1)

i = 0 | tok[i-1] = 'b' <- chính là ký tự CUỐI, không phải 'bên trái'!
0 < i < len(tok)-1 ? False


## 3. Cách 1 — Kiểm tra từng luật bằng `if` (cách trực giác, lời giải chính)

Viết hàm `is_valid(tok)` theo đúng thứ tự 3 luật. Mỗi luật một khối, vi phạm thì `return False` ngay.

### Luật 1 — không có chữ số

In [6]:
def rule_no_digit(tok):
    return not any(ch.isdigit() for ch in tok)

for t in ["cat", "b8d!", "1-s"]:
    print(t, "->", rule_no_digit(t))

cat -> True
b8d! -> False
1-s -> False


### Luật 2 — tối đa một `-`, và hai bên là chữ thường

In [7]:
def rule_hyphen(tok):
    k = tok.count("-")
    if k == 0:
        return True
    if k > 1:
        return False
    i = tok.index("-")
    if i == 0 or i == len(tok) - 1:          # đứng đầu hoặc cuối: thiếu một bên
        return False
    return tok[i - 1].isalpha() and tok[i + 1].isalpha()

for t in ["a-b", "-ab", "ab-", "a--b", "a-b-c", "a-.", "a-b."]:
    print(f"{t:<6} -> {rule_hyphen(t)}")

a-b    -> True
-ab    -> False
ab-    -> False
a--b   -> False
a-b-c  -> False
a-.    -> False
a-b.   -> True


Chú ý `"a-."`: gạch nối không ở cuối token, nhưng bên phải là `.` chứ không phải chữ → sai. Ta dùng `isalpha()` an toàn vì luật 1 đã loại chữ số và đề chỉ có chữ **thường**.

### Luật 3 — tối đa một dấu câu, và phải ở cuối

In [8]:
def rule_punct(tok):
    k = sum(ch in "!.," for ch in tok)
    if k == 0:
        return True
    if k > 1:
        return False
    return tok[-1] in "!.,"                  # dấu câu duy nhất phải là ký tự cuối

for t in ["ab,", "cd!", ".", "a!b", "c.,", "!this"]:
    print(f"{t:<6} -> {rule_punct(t)}")

ab,    -> True
cd!    -> True
.      -> True
a!b    -> False
c.,    -> False
!this  -> False


### Ghép lại

In [9]:
def is_valid(tok):
    return rule_no_digit(tok) and rule_hyphen(tok) and rule_punct(tok)

def count_valid_rules(sentence):
    return sum(1 for tok in sentence.split() if is_valid(tok))

print(count_valid_rules("cat and  dog"))                               # 3
print(count_valid_rules("!this  1-s b8d!"))                            # 0
print(count_valid_rules("alice and  bob are playing stone-game10"))    # 5

3
0
5


`and` trong Python **ngắt mạch** (short-circuit): nếu luật 1 sai thì không chạy luật 2, 3. Thứ tự hợp lý: luật rẻ & loại nhiều nhất đặt trước.

## 4. TRACE từng token

Câu: `"!this  1-s b8d! a-b. a-. x"`.

| token | luật 1 (no digit) | luật 2 (hyphen) | luật 3 (punct) | hợp lệ? |
|---|---|---|---|---|
| `!this` | ✓ | ✓ (0 gạch) | ✗ dấu `!` không ở cuối | ✗ |
| `1-s` | ✗ có `1` | – | – | ✗ |
| `b8d!` | ✗ có `8` | – | – | ✗ |
| `a-b.` | ✓ | ✓ `a`-`b` | ✓ `.` ở cuối | ✓ |
| `a-.` | ✓ | ✗ bên phải là `.` | – | ✗ |
| `x` | ✓ | ✓ | ✓ | ✓ |

Kết quả: 2.

In [10]:
def trace_count(sentence):
    total = 0
    for tok in sentence.split():
        r1 = rule_no_digit(tok)
        r2 = r1 and rule_hyphen(tok)
        r3 = r2 and rule_punct(tok)
        reason = ("có chữ số" if not r1 else
                  "sai luật gạch nối" if not r2 else
                  "sai luật dấu câu" if not r3 else "HỢP LỆ")
        total += r3
        print(f"{tok!r:<9} luật1={r1!s:<5} luật2={r2!s:<5} luật3={r3!s:<5} -> {reason:<17} total={total}")
    return total

trace_count("!this  1-s b8d! a-b. a-. x")

'!this'   luật1=True  luật2=True  luật3=False -> sai luật dấu câu  total=0
'1-s'     luật1=False luật2=False luật3=False -> có chữ số         total=0
'b8d!'    luật1=False luật2=False luật3=False -> có chữ số         total=0
'a-b.'    luật1=True  luật2=True  luật3=True  -> HỢP LỆ            total=1
'a-.'     luật1=True  luật2=False luật3=False -> sai luật gạch nối total=1
'x'       luật1=True  luật2=True  luật3=True  -> HỢP LỆ            total=2


2

`f"{tok!r:<9}"`: `!r` in dạng `repr` (có dấu nháy), `:<9` căn trái độ rộng 9. `total += r3` cộng được vì `True` là 1.

## 5. Nút thắt của cách 1 và ý tưởng cải tiến

Cách 1 đúng, `O(n)` tổng thể, nhưng mỗi token bị duyệt **nhiều lần** (`any`, `count`, `index`, `sum`). Ta có thể:

- **Quét một lần** từng ký tự, cập nhật vài biến trạng thái (đã thấy gạch chưa?), phát hiện vi phạm ngay — cách 3.
- Hoặc mô tả **hình dạng** của từ hợp lệ bằng một mẫu (pattern) — cách 2.

**Insight về hình dạng:** một từ hợp lệ luôn có dạng

```
[chữ+ [ - chữ+ ]]  [dấu câu]
```

tức: (tùy chọn) một khối chữ, có thể nối thêm `-` và một khối chữ nữa; sau đó (tùy chọn) đúng một dấu câu. Nhưng phải có **ít nhất 1 ký tự** — token không rỗng do `split()` đảm bảo.

## 6. Cách 2 — Regex mô tả hình dạng

Ký hiệu regex cần biết:

| Ký hiệu | Nghĩa |
|---|---|
| `[a-z]+` | một hoặc nhiều chữ thường |
| `(...)?` | nhóm tùy chọn (0 hoặc 1 lần) |
| `[!.,]` | một ký tự trong tập `! . ,` |
| `re.fullmatch(p, s)` | khớp **toàn bộ** chuỗi (không chỉ một phần) |

Mẫu: `([a-z]+(-[a-z]+)?)?[!.,]?`

In [11]:
import re
PATTERN = re.compile(r"([a-z]+(-[a-z]+)?)?[!.,]?")

for t in ["a-b.", "afad", "ba-c", "a!", "!", "-ab", "ab-", "a!b", "c.,", "a-.", "a-b-c", "b8d"]:
    print(f"{t:<6} -> {bool(PATTERN.fullmatch(t))}")

a-b.   -> True
afad   -> True
ba-c   -> True
a!     -> True
!      -> True
-ab    -> False
ab-    -> False
a!b    -> False
c.,    -> False
a-.    -> False
a-b-c  -> False
b8d    -> False


In [12]:
def count_valid_regex(sentence):
    return sum(1 for tok in sentence.split() if PATTERN.fullmatch(tok))

print(count_valid_regex("cat and  dog"))
print(count_valid_regex("!this  1-s b8d!"))
print(count_valid_regex("alice and  bob are playing stone-game10"))

3
0
5


Cẩn thận: mẫu trên cũng khớp **chuỗi rỗng** `""` (mọi phần đều tùy chọn). Không sao vì `split()` không bao giờ trả token rỗng — nhưng nếu dùng `split(" ")` thì sẽ đếm sai! Đây là ví dụ tốt cho việc "giả định ngầm phải được ghi rõ".

In [13]:
print(bool(PATTERN.fullmatch("")))                                      # True!
print(sum(1 for t in "cat and  dog".split(" ") if PATTERN.fullmatch(t)))  # 4 (sai)

True
4


## 7. Cách 3 — Máy trạng thái quét một lần mỗi token

Duyệt `i, ch` trong token, chỉ giữ 1 biến `seen_hyphen`:

- `ch` là chữ số → sai.
- `ch == '-'` → sai nếu đã có gạch, hoặc `i == 0`, hoặc `i == len-1`, hoặc hai bên không phải chữ; ngược lại `seen_hyphen = True`.
- `ch` là dấu câu → sai nếu `i != len-1` (không ở cuối). Vì chỉ được ở cuối, tự động "tối đa một".
- chữ thường → luôn ổn.

In [14]:
def is_valid_scan(tok):
    n = len(tok)
    seen_hyphen = False
    for i, ch in enumerate(tok):
        if ch.isdigit():
            return False
        if ch == "-":
            if seen_hyphen or i == 0 or i == n - 1:
                return False
            if not (tok[i - 1].isalpha() and tok[i + 1].isalpha()):
                return False
            seen_hyphen = True
        elif ch in "!.,":
            if i != n - 1:
                return False
    return True

def count_valid_scan(sentence):
    return sum(is_valid_scan(tok) for tok in sentence.split())

print(count_valid_scan("cat and  dog"))
print(count_valid_scan("!this  1-s b8d!"))
print(count_valid_scan("alice and  bob are playing stone-game10"))

3
0
5


Vì sao "dấu câu phải ở cuối" đã ngầm bao "tối đa một dấu câu"? Vị trí cuối chỉ có **một** ô, nên hai dấu câu thì ít nhất một cái không ở cuối → bị loại. Ví dụ `"c.,"`: `.` ở vị trí 1 ≠ 2 → sai.

Trace máy trạng thái trên `"a-b."` và `"a-b-c"`:

In [15]:
def trace_scan(tok):
    n, seen = len(tok), False
    for i, ch in enumerate(tok):
        kind = ("digit" if ch.isdigit() else "hyphen" if ch == "-" else
                "punct" if ch in "!.," else "letter")
        print(f"  i={i} ch={ch!r} loại={kind:<6} seen_hyphen={seen}")
        if kind == "digit":
            print("  -> có chữ số: False"); return False
        if kind == "hyphen":
            if seen or i == 0 or i == n - 1 or not (tok[i-1].isalpha() and tok[i+1].isalpha()):
                print("  -> gạch nối sai vị trí / thứ hai: False"); return False
            seen = True
        if kind == "punct" and i != n - 1:
            print("  -> dấu câu không ở cuối: False"); return False
    print("  -> HỢP LỆ"); return True

for t in ["a-b.", "a-b-c"]:
    print(t); trace_scan(t)

a-b.
  i=0 ch='a' loại=letter seen_hyphen=False
  i=1 ch='-' loại=hyphen seen_hyphen=False
  i=2 ch='b' loại=letter seen_hyphen=True
  i=3 ch='.' loại=punct  seen_hyphen=True
  -> HỢP LỆ
a-b-c
  i=0 ch='a' loại=letter seen_hyphen=False
  i=1 ch='-' loại=hyphen seen_hyphen=False
  i=2 ch='b' loại=letter seen_hyphen=True
  i=3 ch='-' loại=hyphen seen_hyphen=True
  -> gạch nối sai vị trí / thứ hai: False


## 8. Code nộp LeetCode

In [16]:
class Solution:
    def countValidWords(self, sentence: str) -> int:
        def valid(tok: str) -> bool:
            n = len(tok)
            seen_hyphen = False
            for i, ch in enumerate(tok):
                if ch.isdigit():
                    return False
                if ch == "-":
                    if seen_hyphen or i == 0 or i == n - 1:
                        return False
                    if not (tok[i - 1].isalpha() and tok[i + 1].isalpha()):
                        return False
                    seen_hyphen = True
                elif ch in "!.," and i != n - 1:
                    return False
            return True

        return sum(valid(tok) for tok in sentence.split())

print(Solution().countValidWords("alice and  bob are playing stone-game10"))

5


Hàm `valid` lồng trong method (nested function) để gọn — nó chỉ dùng nội bộ. Bản regex cũng nộp được:

In [17]:
class SolutionRegex:
    def countValidWords(self, sentence: str) -> int:
        pat = re.compile(r"([a-z]+(-[a-z]+)?)?[!.,]?")
        return sum(1 for tok in sentence.split() if pat.fullmatch(tok))

print(SolutionRegex().countValidWords("cat and  dog"))

3


## 9. Vì sao đúng?

Với mỗi token khác rỗng, `valid(tok)` trả `True` **khi và chỉ khi** 3 luật đều đúng:

- **(⇒)** Nếu trả `True`, không ký tự nào là chữ số (luật 1). Mỗi `-` gặp phải: chưa từng có `-` trước đó, không ở hai đầu, hai bên là chữ → tối đa một `-` và được bao bởi chữ (luật 2). Mọi dấu câu đều ở vị trí `n-1` → tối đa một và ở cuối (luật 3).
- **(⇐)** Nếu 3 luật đúng thì không nhánh `return False` nào kích hoạt: không có chữ số; gạch nối duy nhất (nếu có) có hai bên là chữ nên không ở đầu/cuối và `seen_hyphen` lúc đó còn `False`; dấu câu duy nhất (nếu có) ở cuối.

Hàm ngoài đếm đúng số token vì `split()` trả **đúng** các token (khối ký tự không phải dấu cách, không rỗng).

## 10. Độ phức tạp

Gọi `n = len(sentence)` (≤ 1000).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Ba luật riêng | `O(n)` (mỗi token duyệt hằng số lần) | `O(n)` cho list token | dễ đọc, dễ sửa từng luật |
| 2. Regex | `O(n)` (mẫu đơn giản, không quay lui bùng nổ) | `O(n)` | ngắn, cần biết regex |
| 3. Máy trạng thái | `O(n)`, một lần quét mỗi token | `O(n)` cho token (`O(1)` thêm) | hiệu quả, dễ chứng minh |

## 11. Bộ test

In [18]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<36} PASS ({actual})")

sol = Solution()
cases = [
    ("ví dụ 1 (2 dấu cách)", "cat and  dog", 3),
    ("ví dụ 2", "!this  1-s b8d!", 0),
    ("ví dụ 3", "alice and  bob are playing stone-game10", 5),
    ("ví dụ từ hợp lệ trong đề", "a-b. afad ba-c a! !", 5),
    ("gạch đầu / cuối", "-ab ab- -", 0),
    ("hai gạch", "a-b-c a--b", 0),
    ("dấu câu giữa / hai dấu", "a!b c., ..", 0),
    ("gạch cạnh dấu câu", "a-. a-!b", 0),
    ("dấu cách đầu/cuối câu", "   hello   world.   ", 2),
    ("một ký tự", "a", 1),
    ("chỉ dấu câu", ",", 1),
    ("chữ số đơn", "5", 0),
]
fs = [count_valid_rules, count_valid_regex, count_valid_scan,
      Solution().countValidWords, SolutionRegex().countValidWords]
for name, s, exp in cases:
    for f in fs:
        assert f(s) == exp, (f, s, f(s), exp)
    check(name, sol.countValidWords(s), exp)

ví dụ 1 (2 dấu cách)                 PASS (3)
ví dụ 2                              PASS (0)
ví dụ 3                              PASS (5)
ví dụ từ hợp lệ trong đề             PASS (5)
gạch đầu / cuối                      PASS (0)
hai gạch                             PASS (0)
dấu câu giữa / hai dấu               PASS (0)
gạch cạnh dấu câu                    PASS (0)
dấu cách đầu/cuối câu                PASS (2)
một ký tự                            PASS (1)
chỉ dấu câu                          PASS (1)
chữ số đơn                           PASS (0)


In [19]:
import random
random.seed(2047)
alphabet = "ab-!.,1 "
for _ in range(3000):
    s = "".join(random.choice(alphabet) for _ in range(random.randint(1, 14)))
    if not s.split():
        continue                          # đề đảm bảo có ít nhất 1 token
    results = {f(s) for f in fs}
    assert len(results) == 1, (s, [f(s) for f in fs])
print("3000 câu ngẫu nhiên: 5 cài đặt cho cùng kết quả — PASS")

3000 câu ngẫu nhiên: 5 cài đặt cho cùng kết quả — PASS


Test ngẫu nhiên với bảng chữ nhỏ `"ab-!.,1 "` sinh ra rất nhiều ca "kỳ quặc" (`"-!"`, `"a-,"`, `"!!"`...) mà ta khó nghĩ ra bằng tay — rất hiệu quả cho bài nhiều luật.

## 12. Lỗi người mới hay gặp

### Lỗi 1: `split(" ")` và đếm token rỗng

In [20]:
def wrong_split_space(sentence):
    return sum(is_valid_scan(t) for t in sentence.split(" "))

print("Sai :", wrong_split_space("cat and  dog"))   # 4: token '' bị coi là hợp lệ
print("Đúng:", count_valid_scan("cat and  dog"))

Sai : 4
Đúng: 3


### Lỗi 2: Kiểm tra hai bên gạch nối mà quên vị trí đầu

In [21]:
def wrong_hyphen_neighbors(tok):
    if tok.count("-") > 1:
        return False
    if "-" in tok:
        i = tok.index("-")
        try:
            return tok[i - 1].isalpha() and tok[i + 1].isalpha()
        except IndexError:
            return False
    return True

print("'-ab' theo hàm sai:", wrong_hyphen_neighbors("-ab"), "(tok[-1]='b' bị coi là bên trái)")
print("'-ab' theo hàm đúng:", rule_hyphen("-ab"))

'-ab' theo hàm sai: True (tok[-1]='b' bị coi là bên trái)
'-ab' theo hàm đúng: False


**Sửa:** kiểm tra `i == 0` (và `i == len-1`) **trước** khi đọc `tok[i-1]`, `tok[i+1]`.

### Lỗi 3: Quên (hoặc kiểm tra hời hợt) luật dấu câu

Nhiều bạn chỉ code luật chữ số và gạch nối, cho rằng "dấu câu thì kiểu gì cũng được". Khi đó `"a!b"` (dấu câu ở giữa) và `"c.,"` (hai dấu câu) đều lọt. Một biến thể khác của lỗi này là chỉ kiểm tra `tok[-1] in "!.,"` — điều đó cho biết **ký tự cuối có là dấu câu không**, chứ không nói gì về các dấu câu nằm ở giữa. Cách kiểm tra đúng: **không có dấu câu nào trong `tok[:-1]`** (mọi vị trí trừ cuối).

### Lỗi 4: Coi token `"-"` hay `"!"` là như nhau

`"!"` **hợp lệ** (một dấu câu ở cuối, không chữ số, không gạch). `"-"` **không hợp lệ** (gạch nối không có chữ hai bên).

In [22]:
def wrong_punct_only_last(tok):
    if any(ch.isdigit() for ch in tok):
        return False
    if not rule_hyphen(tok):
        return False
    return True                              # quên hoàn toàn luật 3

for t in ["a!b", "c.,", "!", "-"]:
    print(f"{t:<4} hàm sai={wrong_punct_only_last(t)!s:<5} hàm đúng={is_valid_scan(t)}")

a!b  hàm sai=True  hàm đúng=False
c.,  hàm sai=True  hàm đúng=False
!    hàm sai=True  hàm đúng=True
-    hàm sai=False hàm đúng=False


## 13. Mẹo nhận dạng pattern

- Đề "**token hợp lệ nếu thỏa các luật sau**" → viết **một hàm `is_valid` riêng**, mỗi luật một khối `if`, `return False` sớm. Đếm bằng `sum(is_valid(t) for t in tokens)`.
- Luật về **vị trí** (đầu, cuối, hai bên) → luôn kiểm tra biên chỉ số trước khi truy cập `i-1`, `i+1`.
- Luật mô tả **hình dạng chuỗi** → cân nhắc regex + `fullmatch`.
- Muốn chắc chắn → **test ngẫu nhiên đối chiếu** hai cài đặt độc lập (luật vs regex).
- Bài tương tự: 65 Valid Number (khó hơn nhiều), 819 Most Common Word, 2042 Check if Numbers Are Ascending in a Sentence, 1592 Rearrange Spaces Between Words.

## 14. Biến thể

1. **Trả về danh sách từ hợp lệ** thay vì số lượng.
2. **Cho phép tối đa k gạch nối** (mỗi gạch vẫn phải bao bởi chữ): regex `[a-z]+(-[a-z]+){0,k}`.
3. **Cho phép chữ hoa:** đổi `[a-z]` thành `[a-zA-Z]`, hoặc chuyển `tok.lower()` trước.

In [23]:
def valid_words(sentence):
    return [t for t in sentence.split() if is_valid_scan(t)]

def count_valid_k_hyphens(sentence, k):
    pat = re.compile(r"([a-z]+(-[a-z]+){0,%d})?[!.,]?" % k)
    return sum(1 for t in sentence.split() if pat.fullmatch(t))

print(valid_words("!this  1-s b8d! a-b. a-. x"))
print(count_valid_k_hyphens("a-b-c x-y q", 1), count_valid_k_hyphens("a-b-c x-y q", 2))

['a-b.', 'x']
2 3


`"...%d..." % k` là cách định dạng chuỗi kiểu cũ: thay `%d` bằng số nguyên `k`. (Không dùng f-string ở đây vì regex chứa dấu `{}` sẽ xung đột.)

## 15. Bài tập tự luyện

**Bài 1 — Đoán output (tính tay):** token nào hợp lệ trong `["x-y!", "x-y-z", "!", "-", "ab.", "a.b", "q8"]`?

**Bài 2 — Đoán output:** `"a  b".split(" ")`, `"a-b".index("-")`, `"abc".find("-")`, `"c.,"[-1]`.

**Bài 3 — TODO:** viết `exercise_is_valid(tok)` theo 3 luật bằng `if`.

**Bài 4 — TODO:** `exercise_count(sentence)` dùng hàm bài 3.

**Bài 5 — TODO biến thể:** `longest_valid_word(sentence)` trả về từ hợp lệ **dài nhất** (nếu bằng nhau lấy từ xuất hiện trước; không có thì `""`).

**Bài 6 — Mini challenge:** `count_valid_one_pass(sentence)` **không dùng `split`**: quét câu một lần, tự nhận biết điểm bắt đầu/kết thúc token, và kiểm tra luật ngay trong lúc quét.

In [24]:
def exercise_is_valid(tok):
    # TODO luật 1: không chữ số
    # TODO luật 2: tối đa 1 '-', không ở đầu/cuối, hai bên là chữ
    # TODO luật 3: dấu câu (nếu có) chỉ ở vị trí cuối
    pass

# assert exercise_is_valid("a-b.") == True
# assert exercise_is_valid("-ab") == False

In [25]:
def exercise_count(sentence):
    # TODO
    pass

# assert exercise_count("cat and  dog") == 3

In [26]:
def longest_valid_word(sentence):
    # TODO
    pass

# assert longest_valid_word("a bb-c d. stone-game10") == "bb-c"
# assert longest_valid_word("1 2 3") == ""

In [27]:
def count_valid_one_pass(sentence):
    # TODO: duyệt sentence + " ", lưu start của token hiện tại
    pass

# assert count_valid_one_pass("alice and  bob are playing stone-game10") == 5

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | áp từng luật | `"x-y-z"` có 2 gạch; `"a.b"` dấu giữa | hợp lệ: `x-y!`, `!`, `ab.` |
| 2 | `split(" ")` giữ token rỗng | `find` trả -1 khi không thấy | `['a','','b']`, `1`, `-1`, `','` |
| 3 | mục 3 | `count`, `index`, `tok[-1]` | `return rule1 and rule2 and rule3` |
| 4 | `split()` không tham số | `sum(...)` | `sum(exercise_is_valid(t) for t in sentence.split())` |
| 5 | lọc từ hợp lệ trước | giữ `best`, cập nhật khi dài hơn **nghiêm ngặt** | `if len(t) > len(best): best = t` |
| 6 | lính canh `" "` cuối | khi gặp `' '` mà `start` khác None → cắt `sentence[start:i]` | gọi `is_valid_scan` trên đoạn cắt |

## 17. Đáp án tham khảo

**Bài 1:** hợp lệ `x-y!`, `!`, `ab.` (3 token). `x-y-z` hai gạch; `-` không có chữ hai bên; `a.b` dấu câu ở giữa; `q8` có chữ số.

**Bài 2:** `['a', '', 'b']` · `1` · `-1` · `','`.

In [28]:
print([t for t in ["x-y!", "x-y-z", "!", "-", "ab.", "a.b", "q8"] if is_valid_scan(t)])
print("a  b".split(" "), "a-b".index("-"), "abc".find("-"), repr("c.,"[-1]))

['x-y!', '!', 'ab.']
['a', '', 'b'] 1 -1 ','


In [29]:
def exercise_is_valid_answer(tok):
    if any(ch.isdigit() for ch in tok):
        return False
    k = tok.count("-")
    if k > 1:
        return False
    if k == 1:
        i = tok.index("-")
        if i == 0 or i == len(tok) - 1:
            return False
        if not (tok[i - 1].isalpha() and tok[i + 1].isalpha()):
            return False
    if any(ch in "!.," for ch in tok[:-1]):    # dấu câu ở bất kỳ đâu trừ vị trí cuối
        return False
    return True

def exercise_count_answer(sentence):
    return sum(exercise_is_valid_answer(t) for t in sentence.split())

def longest_valid_word_answer(sentence):
    best = ""
    for t in sentence.split():
        if exercise_is_valid_answer(t) and len(t) > len(best):
            best = t
    return best

def count_valid_one_pass_answer(sentence):
    total, start = 0, None
    for i, ch in enumerate(sentence + " "):
        if ch != " " and start is None:
            start = i                              # bắt đầu token mới
        elif ch == " " and start is not None:
            total += is_valid_scan(sentence[start:i])
            start = None                           # kết thúc token
    return total

assert exercise_is_valid_answer("a-b.") and not exercise_is_valid_answer("-ab")
assert exercise_count_answer("cat and  dog") == 3
assert longest_valid_word_answer("a bb-c d. stone-game10") == "bb-c"
assert longest_valid_word_answer("1 2 3") == ""
assert count_valid_one_pass_answer("alice and  bob are playing stone-game10") == 5
random.seed(3)
for _ in range(2000):
    s = "".join(random.choice(alphabet) for _ in range(random.randint(1, 14)))
    if s.split():
        assert exercise_count_answer(s) == count_valid_one_pass_answer(s) == count_valid_scan(s), s
print("Tất cả đáp án bài tập — PASS")

Tất cả đáp án bài tập — PASS


## 18. Tổng kết — template + checklist

**Template "đếm token thỏa nhiều luật":**

```python
def valid(tok):
    n = len(tok)
    for i, ch in enumerate(tok):
        # mỗi luật -> một nhánh if, vi phạm thì return False
        # luật vị trí: kiểm tra i == 0 / i == n-1 TRƯỚC khi đọc tok[i-1], tok[i+1]
        ...
    return True

return sum(valid(t) for t in s.split())     # split() KHÔNG tham số
```

Regex nhớ nhanh cho bài này: `([a-z]+(-[a-z]+)?)?[!.,]?` + `fullmatch`.

**Checklist submit**

- [ ] Dùng `split()` không tham số (nhiều dấu cách, không token rỗng).
- [ ] Luật 1: loại mọi token có chữ số.
- [ ] Luật 2: ≤ 1 gạch, không ở đầu/cuối, hai bên là chữ (cẩn thận `tok[-1]` khi `i = 0`).
- [ ] Luật 3: dấu câu chỉ ở vị trí cuối (tự động ≤ 1).
- [ ] Đã thử `"-"`, `"!"`, `"a-."`, `"a!b"`, `"c.,"`.